# Hướng dẫn Chạy mô hình Multimodal (Late Fusion) trên Google Colab
Notebook này đã được cấu hình tự động 100% để bạn có thể chạy từ đầu đến cuối mà không gặp lỗi.

**LƯU Ý TRƯỚC KHI CHẠY:**
Bạn cần đảm bảo đã tạo thư mục Lối tắt (Shortcut) tên là `SE365` trên Google Drive của bạn (trỏ từ link chia sẻ dữ liệu của nhóm). Nếu bạn đặt tên lối tắt khác, hãy sửa tên thư mục ở **Bước 3** và **Bước 3.5**

### BƯỚC 1: Mount Google Drive
Lệnh này sẽ yêu cầu bạn cấp quyền truy cập Google Drive. Chúng ta cần làm điều này để đọc dữ liệu gốc (5000 ảnh và CSV) thông qua Lối tắt (Shortcut) mà không cần tải lại file zip.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### BƯỚC 2: Clone mã nguồn và cài đặt thư viện
Tải phiên bản code mới nhất từ Github và cài đặt các thư viện cần thiết (PyTorch, Transformers, Timm, ...).


In [2]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt

Cloning into 'SE365'...
remote: Enumerating objects: 13121, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 13121 (delta 33), reused 55 (delta 22), pack-reused 13047 (from 2)
Receiving objects: 100% (13121/13121), 871.76 MiB | 31.50 MiB/s, done.
Resolving deltas: 100% (210/210), done.
/content/SE365


### BƯỚC 3: Nạp dữ liệu vào máy ảo Colab bằng gdown (QUAN TRỌNG)
Chúng ta sẽ tải trực tiếp file nén `data.zip` từ Google Drive thông qua gdown. Bạn cần upload `data.zip` lên Drive, chuột phải chọn Share -> Anyone with the link, sau đó copy File ID (đoạn mã dài trên URL).

**Chỗ cần sửa:** Dán File ID của bạn thay cho chuỗi `YOUR_FILE_ID` bên dưới.

In [3]:
!rm -rf ./data
# SỬA LẠI FILE ID NẾU CẦN
!gdown --id 11WoeUn2visKtGN5oOX9c2I6Grz3P88vD -O data.zip

# Giải nén data.zip (mặc định zip không làm giảm chất lượng ảnh)
!unzip -q data.zip

# Xoá file nén sau khi giải xong để nhẹ ổ cứng
!rm data.zip

# Kiểm tra xem dữ liệu đã được nạp chưa
!ls -la ./data

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=11WoeUn2visKtGN5oOX9c2I6Grz3P88vD
From (redirected): https://drive.google.com/uc?id=11WoeUn2visKtGN5oOX9c2I6Grz3P88vD&confirm=t&uuid=63397420-fb9a-4de8-bbb4-d76cff8566b5
To: /content/SE365/data.zip
100% 4.02G/4.02G [01:19<00:00, 50.6MB/s]
total 1372
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 16 13:58 ..
drwxr-xr-x  2 root root 1388544 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### BƯỚC 3.5: Cấu hình nơi lưu Checkpoint (Trọng số mô hình)
Để tránh bị mất kết quả khi Colab tự ngắt, đoạn code sau sẽ tạo một thư mục con trong `checkpoints/` trên Drive của bạn, với tên là thời gian hiện tại (Ví dụ: `20260611_083908`).
Tất cả các mô hình Text, Image, Fusion sau khi train xong sẽ được tự động copy vào chung thư mục này.

**Chỗ cần sửa:** Nếu bạn muốn lưu vào thư mục khác, hãy sửa biến `drive_ckpt_path`.


In [4]:
# BƯỚC 3.5: Khởi tạo thư mục lưu trữ cho toàn bộ phiên chạy này
# Đảm bảo tất cả mô hình train trong hôm nay đều nằm chung một thư mục
import os
import datetime

run_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
drive_ckpt_path = f'/content/drive/MyDrive/SE365/checkpoints/{run_id}'
os.environ['DRIVE_CKPT'] = drive_ckpt_path

# Tạo thư mục con checkpoints và thư mục run_id bằng os.makedirs
os.makedirs(drive_ckpt_path, exist_ok=True)
print(f'Mọi checkpoint trong phiên này sẽ được lưu chung vào: {drive_ckpt_path}')

Mọi checkpoint trong phiên này sẽ được lưu chung vào: /content/drive/MyDrive/SE365/checkpoints/20260616_135801


### BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
**Các tham số bạn có thể tuỳ chỉnh ở lệnh Train (cả 3 mô hình text, imame, fusion):**

Các tham số liên quan đường dẫn
- `--train_path`: Đường dẫn đến file train.csv *(Mặc định: `./data/text/train.csv`)*
- `--val_path`: Đường dẫn đến file val.csv *(Mặc định: `./data/text/val.csv`)*
- `--test_path`: Đường dẫn đến file test.csv *(Mặc định: `./data/text/test.csv`)*
- `--image_dir`: Đường dẫn đến thư mục ảnh *(Mặc định: `./data/image`)*

Các tham số liên quan huấn luyện mô hình
- `--mode`: Chế độ chạy (`train_text`, `train_image`, `train_fusion`)
- `--epochs`: Số vòng lặp (Mặc định: 5)
- `--batch_size`: Kích thước batch (Mặc định: 16)
- `--lr`: Learning rate (Mặc định: 2e-5)
- `--alpha`: Trọng số loss cho các yếu tố phụ (Mặc định: 0.5)

In [5]:
# BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
!python main.py --mode train_text --epochs 15 --grad_accum_steps 4

# Lưu Checkpoint Text ngay lập tức
!cp ./checkpoints/best_model_train_text.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Text vào $DRIVE_CKPT"

====== MODE: TRAIN_TEXT ======
Using device: cuda
config.json: 100% 615/615 [00:00<00:00, 590kB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 90.2kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 28.5MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 21.7MB/s]
preprocessor_config.json: 100% 266/266 [00:00<00:00, 1.45MB/s]
Loading Dataset...
Đã nạp 4800 mẫu cho Train và 600 mẫu cho Val
Initializing Model...
model.safetensors: 100% 1.12G/1.12G [00:07<00:00, 151MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 25210.42it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

### BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)

In [12]:
# BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)
!python main.py --mode train_image --epochs 15 --batch_size 8 --grad_accum_steps 2

# Lưu Checkpoint Image ngay lập tức
!cp ./checkpoints/best_model_train_image.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Image vào $DRIVE_CKPT"

Exception ignored in: <function _get_module_lock.<locals>.cb at 0x7de01b363880>
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 451, in cb
KeyboardInterrupt: 
Traceback (most recent call last):
  File "/content/SE365/main.py", line 1, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2804, in <module>
    from torch import _meta_registrations
  File "/usr/local/lib/python3.12/dist-packages/torch/_meta_registrations.py", line 12, in <module>
    from torch._decomp import (
  File "/usr/local/lib/python3.12/dist-packages/torch/_decomp/__init__.py", line 287, in <module>
    import torch._decomp.decompositions
  File "/usr/local/lib/python3.12/dist-packages/torch/_decomp/decompositions.py", line 17, in <module>
    import torch._prims as prims
  File "/usr/local/lib/python3.12/dist-packages/torch/_prims/__init__.py", line 2877, in <module>
    _uniform_helper = _make_prim(
                      ^^^^^^^^^^^

### BƯỚC 6: Huấn luyện mô hình Fusion (Kết hợp Text + Image)
Mô hình này sẽ tự động tải các checkpoint tốt nhất của mô hình Text và Image vừa train ở trên để tiếp tục huấn luyện phần kết hợp.

In [13]:
# BƯỚC 6: Huấn luyện mô hình Fusion (Kết hợp Text + Image)
!python main.py --mode train_fusion --epochs 10 --grad_accum_steps 4 --unfreeze_text_layers 1 --unfreeze_image_layers 1

# Lưu Checkpoint Fusion ngay lập tức
!cp ./checkpoints/best_model_train_fusion.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Fusion vào $DRIVE_CKPT"

====== MODE: TRAIN_FUSION ======
Using device: cuda
Loading Dataset...
Đã nạp 4800 mẫu cho Train và 600 mẫu cho Val
Initializing Model...
Loading weights: 100% 199/199 [00:00<00:00, 9582.08it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(
Loaded Text Model weights from ./checkpoints/best_model_train_text.pth
Loaded Image Model weights from ./checkpoint

### BƯỚC 7: Đánh giá (Test) trên mô hình tốt nhất
Bước cuối cùng là chạy đánh giá mô hình Fusion trên tập dữ liệu Test chưa từng được thấy để tính ra các chỉ số sai số MAE, MSE và RMSE.

**Các tham số bạn có thể tuỳ chỉnh ở lệnh Test:** Khác với Train, lệnh Test chỉ quan tâm tới các tham số: `--test_path`, `--image_dir` và `--batch_size`.

In [14]:
# BƯỚC 7: Đánh giá mô hình (Tính các metric MAE, MSE, RMSE)
!python test.py --mode train_fusion

====== TESTING MODE: TRAIN_FUSION ======
Loading Test Dataset...
Đã nạp 600 mẫu Test
Loading weights: 100% 199/199 [00:00<00:00, 760.61it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(

[EVALUATION METRICS ON INDEPENDENT TEST SET]
MSE  | Food: 45.6432 | Price: 44.7292 | Atmos: 39.9962 | Service: 51.8820 | Overall: 44.1851
RMSE | Food: 6.7560 | Price: 6